## Match to OpenAlex

In [3]:
import requests
from fuzzywuzzy import fuzz
import time
import json
import pandas as pd

def search_openalex_by_title(title, year, min_score=85):
    """
    Search OpenAlex for a work by title and year.
    Returns (work_dict, confidence_score) or (None, best_score_found)
    """
    url = "https://api.openalex.org/works"
    params = {
        "filter": f"publication_year:{year}",
        "search": title,
        "per-page": 5,
        "mailto": "shaheryar.4822@student.uu.se"  # Polite pooling
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code != 200:
            return None, 0
        
        results = response.json().get("results", [])
        
        # Fuzzy match titles
        best_match = None
        best_score = 0
        
        for work in results:
            work_title = work.get("display_name", "")
            score = fuzz.ratio(title.lower(), work_title.lower())
            
            if score > best_score:
                best_score = score
                best_match = work
        
        if best_score >= min_score:
            return best_match, best_score
        else:
            return None, best_score
            
    except Exception as e:
        print(f"   ⚠️  API Error: {e}")
        return None, 0

# Load awards
df_scope = pd.read_csv("huang_awards_pilot.csv")

# Match all awards
matched_awards = []
unmatched_awards = []

print(f"\n{'='*60}")
print(f"MATCHING {len(df_scope)} AWARDS TO OPENALEX")
print(f"{'='*60}\n")

for idx, row in df_scope.iterrows():
    conf = row['conference']
    year = int(row['year'])
    title = row['paper_title']
    
    print(f"[{idx+1}/{len(df_scope)}] {conf} {year}: {title[:60]}...")
    
    # Search OpenAlex
    work, score = search_openalex_by_title(title, year)
    
    if work:
        # Matched!
        matched_awards.append({
            'award_id': idx,
            'conference': conf,
            'year': year,
            'award_title': title,
            'award_authors': row['authors'],
            'award_url': row.get('paper_url', ''),
            'match_score': score,
            'openalex_id': work['id'],
            'openalex_title': work['display_name'],
            'cited_by_count': work['cited_by_count'],
            'publication_date': work.get('publication_date', ''),
            'authorships': json.dumps(work['authorships'])  # Store as JSON string
        })
        print(f"  ✅ Match score: {score}")
    else:
        # Unmatched
        unmatched_awards.append({
            'award_id': idx,
            'conference': conf,
            'year': year,
            'award_title': title,
            'best_score': score
        })
        print(f"  ❌ No match (best score: {score})")
    
    # Rate limiting
    time.sleep(0.2)
    
    # Save progress every 50 awards
    if (idx + 1) % 50 == 0:
        pd.DataFrame(matched_awards).to_csv("matched_awards_progress.csv", index=False)
        print(f"\n💾 Progress saved: {len(matched_awards)} matched so far\n")

# Final save
matched_df = pd.DataFrame(matched_awards)
unmatched_df = pd.DataFrame(unmatched_awards)

matched_df.to_csv("matched_awards_2010_2018.csv", index=False)
unmatched_df.to_csv("unmatched_awards_2010_2018.csv", index=False)

# Summary
print(f"\n{'='*60}")
print(f"MATCHING COMPLETE")
print(f"{'='*60}")
print(f"✅ Successfully matched: {len(matched_df)}/{len(df_scope)} ({100*len(matched_df)/len(df_scope):.1f}%)")
print(f"❌ Unmatched: {len(unmatched_df)}")
print(f"Average match score: {matched_df['match_score'].mean():.1f}")
print(f"\nMatch rate by conference:")
print(matched_df.groupby('conference').size())
print(f"\nMatch rate by year:")
print(matched_df.groupby('year').size())



MATCHING 411 AWARDS TO OPENALEX

[1/411] CHI 2018: Agile 3D Sketching with Air Scaffolding...
  ✅ Match score: 100
[2/411] CHI 2018: Pinpointing: Precise Head- and Eye-Based Target Selection fo...
  ❌ No match (best score: 36)
[3/411] CHI 2018: Data Illustrator: Augmenting Vector Design Tools with Lazy D...
  ❌ No match (best score: 25)
[4/411] CHI 2018: "A Stalker's Paradise": How Intimate Partner Abusers Exploit...
  ❌ No match (best score: 19)
[5/411] CHI 2018: Keeping a Low Profile? Technology, Risk and Privacy among Un...
  ❌ No match (best score: 43)
[6/411] CHI 2018: Streets for People: Engaging Children in Placemaking Through...
  ❌ No match (best score: 35)
[7/411] CHI 2018: Addressing Age-Related Bias in Sentiment Analysis...
  ✅ Match score: 100
[8/411] CHI 2018: From Her Story, to Our Story: Digital Storytelling as Public...
  ❌ No match (best score: 39)
[9/411] CHI 2018: Gender Recognition or Gender Reductionism?: The Social Impli...
  ❌ No match (best score: 31)
[10/411]